# 01 - Inspeccion inicial del dataset

Este notebook documenta la primera inspeccion del dataset de reservas hoteleras. El objetivo no es hacer todavia un EDA completo, sino confirmar que el archivo existe, entender sus columnas, validar el target y dejar registradas las primeras decisiones tecnicas.

Proyecto: clasificacion de reservas hoteleras.

Cliente hipotetico: plataforma de reservas tipo Booking, Trivago o Agoda.

Pregunta de negocio: podemos anticipar si una reserva sera cancelada?

## 1. Importacion de librerias

Usamos `pandas` porque permite leer archivos tabulares y revisar datos de forma clara. Tambien usamos `Path` para construir rutas de forma mas robusta.

In [ ]:
from pathlib import Path

import pandas as pd

## 2. Ruta del dataset

El CSV descargado desde Kaggle esta ubicado dentro de `data/raw/`. Mantenemos los datos crudos separados para no mezclar el archivo original con datos transformados.

In [ ]:
DATA_PATH = Path("../data/raw/hotel-reservations-classification-dataset/Hotel Reservations.csv")

DATA_PATH.exists(), DATA_PATH

## 3. Carga del dataset

Cargamos el CSV en un DataFrame llamado `df`. En este paso no modificamos datos: solo leemos el archivo original.

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head(10)

## 4. Dimensiones del dataset

Revisamos cuantas filas y columnas tiene el dataset. Esto nos ayuda a saber si hay volumen suficiente para separar datos de entrenamiento, validacion y test.

In [ ]:
n_rows, n_columns = df.shape

print(f"Filas: {n_rows}")
print(f"Columnas: {n_columns}")

Interpretacion inicial esperada:

- El dataset tiene 36.275 filas y 19 columnas.
- Es un tamano suficiente para una primera solucion de clasificacion supervisada.
- Mas adelante separaremos los datos en train, validacion y test.

## 5. Columnas disponibles

Listamos las columnas para identificar posibles features, target y columnas que no deberian entrar al modelo.

In [ ]:
df.columns.tolist()

Primera decision:

- `booking_status` sera el target.
- `Booking_ID` se excluira inicialmente porque es un identificador y no deberia generalizar para nuevas reservas.
- El resto de columnas son candidatas iniciales a features, pendiente de revisar tipos, nulos, valores raros y posible leakage.

## 6. Target: `booking_status`

El target es la columna que queremos predecir. En este proyecto queremos saber si una reserva sera cancelada o no, por lo que el problema es de clasificacion binaria.

In [ ]:
TARGET = "booking_status"

df[TARGET].value_counts(dropna=False)

In [ ]:
df[TARGET].value_counts(normalize=True, dropna=False).round(4)

Interpretacion inicial confirmada:

- `Not_Canceled`: 24.390 reservas, aproximadamente 67.2%.
- `Canceled`: 11.885 reservas, aproximadamente 32.8%.
- No hay nulos en el target.

Esto muestra un desbalance moderado. Por eso no deberiamos evaluar el modelo solo con `accuracy`; tambien necesitaremos `precision`, `recall`, `F1-score`, matriz de confusion y ROC-AUC si aplica.

## 7. Tipos de datos iniciales

Antes de limpiar o transformar, revisamos como interpreta pandas cada columna. Esto nos ayuda a separar variables numericas y categoricas.

In [ ]:
df.info()

## 8. Nulos por columna

Los valores nulos pueden requerir imputacion o decisiones de limpieza. En esta etapa solo los contamos.

In [ ]:
df.isna().sum().sort_values(ascending=False)

## 9. Features candidatas iniciales

Estas son las primeras columnas candidatas para entrenar el modelo. La lista puede cambiar despues del EDA.

In [ ]:
initial_features = [
    "no_of_adults",
    "no_of_children",
    "no_of_weekend_nights",
    "no_of_week_nights",
    "type_of_meal_plan",
    "required_car_parking_space",
    "room_type_reserved",
    "lead_time",
    "arrival_year",
    "arrival_month",
    "arrival_date",
    "market_segment_type",
    "repeated_guest",
    "no_of_previous_cancellations",
    "no_of_previous_bookings_not_canceled",
    "avg_price_per_room",
    "no_of_special_requests",
]

excluded_columns = ["Booking_ID"]

print(f"Features candidatas: {len(initial_features)}")
print(f"Columnas excluidas inicialmente: {excluded_columns}")

## 10. Conclusiones de la inspeccion inicial

- El dataset fue cargado correctamente.
- El problema es de clasificacion supervisada binaria.
- El target sera `booking_status`.
- La columna `Booking_ID` queda excluida inicialmente.
- El target tiene dos clases: `Not_Canceled` y `Canceled`.
- La clase mayoritaria es `Not_Canceled`, por lo que hay desbalance moderado.
- Las metricas deberan incluir mas que `accuracy`.

Proximo paso: construir un diccionario inicial de datos y empezar el EDA exploratorio.